<!--
Copyright 2026 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
-->

# **The Compilation Cache**

Understanding "Cold Starts" vs. "Warm Hits" is essential for achieving peak TPU performance. This guide explains how graph fingerprints are generated, why compilation storms happen, and how to ensure your training loop stays on the fast path.

## **How the Cache Works**

The first time a specific sequence of operations is executed, the backend must compile it into TPU machine code. This is a **"Cold Start"** and for large models, can add up to several seconds or even minutes.

Once compiled, the resulting binary is stored in the **Compilation Cache**. The second time you run same sequence of ops (e.g., iteration 2 of a training loop), it will be a **"Warm Hit"** and execution starts instantly.

```
Iteration 1:  Record Graph → Fingerprint → Cache MISS → Compile → Execute → Store in Cache
Iteration 2:  Record Graph → Fingerprint → Cache HIT  → Execute (instant)
Iteration 3:  Record Graph → Fingerprint → Cache HIT  → Execute (instant)
...
```

## **Graph Fingerprinting**

The backend generates a **fingerprint** for each computation graph based on:

1. **Input shapes** — e.g., `[32, 128]` vs. `[16, 128]` Note that **input scalar** values also affect the fingerprint.
2. **Input dtypes** — e.g., `float32` vs. `bfloat16`
3. **Sequence of operations** — the exact ATen ops in order

If **any** of these change, the fingerprint most likely changes, and the cache produces a **miss** — triggering a full compilation.

## **Monitoring Cache Performance**

You can audit your cache performance directly from Python to ensure your training loop is hitting the "Warm" path.

In [2]:
import torch

device = torch.device("tpu")

# Run some operations to populate the cache
x = torch.randn(32, 128, device="tpu")
y = x @ x.T
_ = y.cpu()  # Trigger materialization → compilation

# Run the same graph again (should be a cache hit)
x2 = torch.randn(32, 128, device="tpu")
y2 = x2 @ x2.T
_ = y2.cpu()

# Fetch cache statistics
cache_stats = torch.tpu._get_cache_stats()
print(f"Total Requests: {cache_stats.num_cache_reqs}")
print(f"Total Hits:     {cache_stats.num_cache_hits}")
print(
    "Hit Rate:      "
    f" {cache_stats.num_cache_hits / max(cache_stats.num_cache_reqs, 1) * 100:.1f}%"
)

# Inspect individual cache entries
for i, entry in enumerate(cache_stats.per_entry_stats):
  print(f"\nEntry {i}:")
  print(f"  Read Count: {entry.read_count}")
  print(f"  Compilation Duration (us): {entry.compilation_duration}")
  print(f"  Last Read: {entry.last_read}")

I0000 00:00:1787340657.912158   77223 pjrt_api.cc:119] GetPjrtApi was found for TPU at /home/mateusz/miniforge3/envs/tpu-experiment/lib/python3.12/site-packages/libtpu/libtpu.so
I0000 00:00:1787340657.928582   77223 tpu_profiler_plugin.cc:791] PyTorch PrivateUse1ProfilerRegistry not found. Native TPU profiling will be disabled (this is expected on PyTorch < 2.12).
Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
I0000 00:00:1787340657.949776   77223 device_rt.cc:130] PjRt runtime initialization deferred for tpu
I0000 00:00:1787340657.951398   77223 pjrt_api.cc:168] The PJRT plugin has PJRT API version 0.114. The framework PJRT API version is 0.114.


Device type: tpu, Device index: default
Initializing TPU distributed runtime


I0000 00:00:1787340659.268480   77223 pjrt_c_api_client.cc:188] PjRtCApiClient created.
I0000 00:00:1787340659.284785   77458 compilation_cache_utils.cc:217] Tier-3 compilation cache root directory: 
I0000 00:00:1787340659.284820   77458 tier2_compilation_cache.cc:82] Tier-2 compilation cache is disabled for world size 1.
I0000 00:00:1787340659.284824   77458 tier3_compilation_cache.cc:50] Backup compilation for tier-3 cache read is disabled as tier-3 cache is disabled.


Total Requests: 6
Total Hits:     3
Hit Rate:       50.0%

Entry 0:
  Read Count: 2
  Compilation Duration (us): 0:00:00.316285
  Last Read: 2026-08-21 19:30:59.617227

Entry 1:
  Read Count: 2
  Compilation Duration (us): 0:00:00.329227
  Last Read: 2026-08-21 19:30:59.617020

Entry 2:
  Read Count: 2
  Compilation Duration (us): 0:00:00.320775
  Last Read: 2026-08-21 19:30:59.617073


## **Avoiding Unnecessary Recompilations**
If your input shapes change every iteration (e.g., variable sequence lengths without padding), or if you have data-dependent operations (e.g. nonzero()), the backend may generate a new fingerprint and trigger a re-compilation of the changed ops.

| Scenario | Fingerprints | Performance |
| :--- | :--- | :--- |
| Fixed batch size (32), fixed sequence length (128) | 1 unique graph | ✅ Fast — always cache hits |
| Fixed batch size (32), 3 sequence buckets (128, 512, 1024) | 3 unique graphs | ✅ Good — 3 cold starts, then all hits |
| Variable sequence lengths (every batch different) | N unique graphs | ❌ Re-compilation |

A common cause of recompilation in production is an inconsistent batch size.

* **Scenario:** Batch size = 32, dataset has 100 samples. Batches 1–3 are size 32, but batch 4 is size 4.
* **Result:** The shape change (32 → 4) triggers re-compilation on the last step of **every epoch**.
* **Solution:** Use `drop_last=True` in your DataLoader.

```python
train_loader = torch.utils.data.DataLoader(
    dataset, batch_size=32, shuffle=True,
    drop_last=True  # CRITICAL for TPU performance
)
```

## **Warmup Runs**

When measuring step time, make sure to exclude the first few steps as that can include the warmup (compilation) phase.

**Recommendation:** Run 2 to 3 warmup iterations before starting your timer.

```python
# Warmup iterations to populate the cache
for _ in range(2):
    output = model(warmup_data)
    output.cpu()  # Force materialization → compilation

# Now start measuring real performance
start_time = time.time()
# ... actual training loop ...
```

Use the above compilation cache statistics API after a handful of steps to ensure there are no more recompilations.

> [!TIP]
> Use `torch.tpu._hbm_usage_summary()` to check how much HBM is consumed by cached compiled binaries.